# Feature Selection, Leakage Removal & Feature Engineering — Member 2

**Course:** IT3051 – Fundamentals of Data Mining | **Project:** Telco Customer Churn Prediction

**Input:** `cleaned_data.csv` (output of Member 1 – Data Cleaning & Data Quality)
**Output:** `feature_selected_data.csv` (input for Member 3 – Encoding, Scaling & Train/Test Pipeline)

**Objective:** decide which variables are legitimate predictors that would be available *before* churn
happens, remove identifiers / leakage / redundant variables, and create justified derived features.

**Guiding principle:** a feature may only be used if it would be known at the moment the business wants
to predict *future* churn. Any variable that is generated after, or as a consequence of, the churn
outcome must not enter the model.

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("cleaned_data.csv")

print("Rows, columns:", df.shape)

Rows, columns: (7043, 50)


## 1. Target variable

`Churn Label` is the binary target: `Yes` = customer churned, `No` = customer did not churn.

In [2]:
TARGET = "Churn Label"

print(df[TARGET].value_counts())
print()
print((df[TARGET].value_counts(normalize=True) * 100).round(2))

# numeric copy of the target, used only for the correlation checks in this notebook
y = (df[TARGET] == "Yes").astype(int)

Churn Label
No     5174
Yes    1869
Name: count, dtype: int64

Churn Label
No     73.46
Yes    26.54
Name: proportion, dtype: float64


## 2. Data leakage review

A leakage variable is one that is only known **because** the outcome already happened, or that encodes
the outcome directly. Training on such a variable produces an unrealistically good model that cannot be
reproduced in deployment, because the value simply does not exist for a customer who has not churned yet.

The checks below provide the evidence for each removal decision.

In [3]:
# 2.1 Customer Status vs the target
print(pd.crosstab(df["Customer Status"], df[TARGET]))

Churn Label        No   Yes
Customer Status            
Churned             0  1869
Joined            454     0
Stayed           4720     0


In [4]:
# 2.2 Churn Category / Churn Reason are only populated for customers who already churned
for col in ["Churn Category", "Churn Reason"]:
    print(col, "- missing:", df[col].isnull().sum(), "| missing among non-churners:",
          df.loc[df[TARGET] == "No", col].isnull().sum())

Churn Category - missing: 5174 | missing among non-churners: 5174
Churn Reason - missing: 5174 | missing among non-churners: 5174


In [5]:
# 2.3 Churn Score is a churn-risk score produced from churn information
print("Correlation of Churn Score with the target:", round(np.corrcoef(df["Churn Score"], y)[0, 1], 3))
print()
print(df.groupby(TARGET)["Churn Score"].describe()[["mean", "min", "max"]].round(2))

Correlation of Churn Score with the target: 0.661

              mean   min   max
Churn Label                   
No           50.10   5.0  80.0
Yes          81.78  65.0  96.0


In [6]:
# 2.4 Satisfaction Score - checked because the brief allows it "if justified by project scope"
ct = pd.crosstab(df["Satisfaction Score"], df[TARGET])
ct["churn_rate_%"] = (ct["Yes"] / (ct["Yes"] + ct["No"]) * 100).round(1)
print(ct)
print()
print("Correlation of Satisfaction Score with the target:",
      round(np.corrcoef(df["Satisfaction Score"], y)[0, 1], 3))

Churn Label           No  Yes  churn_rate_%
Satisfaction Score                         
1                      0  922         100.0
2                      0  518         100.0
3                   2236  429          16.1
4                   1789    0           0.0
5                   1149    0           0.0

Correlation of Satisfaction Score with the target: -0.755


### 2.5 Leakage decisions

| Column | Decision | Justification (evidence above) |
|---|---|---|
| `Churn Label` | **TARGET (y)** | The binary variable the project predicts. |
| `Customer ID` | Remove | Unique identifier (7,043 distinct values); carries no predictive information and would let a model memorise rows. |
| `Customer Status` | Remove | Crosstab shows *Churned → all Yes* and *Joined/Stayed → all No*: it is the target under a different name. |
| `Churn Category` | Remove | Populated only for the 1,869 churned customers; all 5,174 non-churners are missing. Post-outcome information. |
| `Churn Reason` | Remove | Same missingness pattern; it explains *why* a customer churned, so it exists only after churn. |
| `Churn Score` | Remove | A churn-risk score derived from churn information (corr = 0.661 with the target); not a raw customer attribute. |
| `Satisfaction Score` | Remove | Scores 1–2 are **100% churners** and scores 4–5 are **0% churners** (corr = −0.755). The survey result separates the classes almost deterministically, which means it reflects the exit rather than predicts it. Keeping it would inflate every model metric and hide the effect of the real business drivers. |

> **Note on Satisfaction Score:** the assignment document lists it as a candidate *"if justified by project
> scope"*. The evidence above shows it is not defensible as a pre-churn predictor, so it is removed and the
> removal is recorded here. If the group later decides to keep it, it must be reported as a
> known-optimistic feature.

In [7]:
leakage_cols = [
    "Customer ID",        # identifier
    "Customer Status",    # encodes the target
    "Churn Category",     # post-outcome
    "Churn Reason",       # post-outcome
    "Churn Score",        # derived from churn information
    "Satisfaction Score"  # near-deterministic separation of the classes
]

print("Columns removed for identifier / leakage reasons:", len(leakage_cols))
for c in leakage_cols:
    print(" -", c)

Columns removed for identifier / leakage reasons: 6
 - Customer ID
 - Customer Status
 - Churn Category
 - Churn Reason
 - Churn Score
 - Satisfaction Score


## 3. Zero-variance columns

A column with a single distinct value cannot help any model separate churners from non-churners.

In [8]:
constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) == 1]

for c in constant_cols:
    print(f"{c}: {df[c].nunique()} unique value -> {df[c].unique()[0]}")

Country: 1 unique value -> United States
State: 1 unique value -> California
Quarter: 1 unique value -> Q3


`Country`, `State` and `Quarter` are constant across all 7,043 records (this is a single-country,
single-quarter extract), so they are removed.

## 4. Geographic variables

The brief asks for geographic variables to be reviewed for usefulness and dimensionality before inclusion.
Two things are checked: how many categories one-hot encoding would create, and how much the variable
actually relates to churn.

In [9]:
geo_cols = ["City", "Zip Code", "Latitude", "Longitude", "Population"]

for c in geo_cols:
    corr = np.corrcoef(df[c], y)[0, 1] if pd.api.types.is_numeric_dtype(df[c]) else np.nan
    print(f"{c:<12} unique values: {df[c].nunique():>5}   correlation with target: "
          f"{'n/a (text)' if np.isnan(corr) else round(corr, 3)}")

City         unique values:  1106   correlation with target: n/a (text)
Zip Code     unique values:  1626   correlation with target: -0.016
Latitude     unique values:  1626   correlation with target: -0.042
Longitude    unique values:  1625   correlation with target: 0.024
Population   unique values:  1569   correlation with target: 0.052


**Decision — remove all five geographic columns.**

* `City` has 1,106 categories and `Zip Code` 1,626. One-hot encoding them would add thousands of sparse
  columns to a 7,043-row dataset, which invites overfitting and makes the model impossible to explain in
  the viva.
* `Latitude`, `Longitude`, `Zip Code` and `Population` all have correlations with churn of |r| ≤ 0.06,
  i.e. effectively no linear relationship.
* They are also mutually redundant (Zip Code ↔ Latitude r = 0.895, Latitude ↔ Longitude r = −0.886) —
  they are three encodings of the same location.

The company operates in one state, so location is not a plausible churn driver here; contract, service and
billing behaviour are.

## 5. Redundant and duplicated variables

Several columns are exact functions of another column. Keeping both adds multicollinearity without adding
information, so the richer version of each pair is kept.

In [10]:
# 5.1 Binary flags that are deterministic recodes of another column
print("Under 30 vs (Age < 30)")
print(pd.crosstab(df["Under 30"], df["Age"] < 30), "\n")

print("Senior Citizen vs (Age >= 65)")
print(pd.crosstab(df["Senior Citizen"], df["Age"] >= 65), "\n")

print("Dependents vs (Number of Dependents > 0)")
print(pd.crosstab(df["Dependents"], df["Number of Dependents"] > 0), "\n")

print("Referred a Friend vs (Number of Referrals > 0)")
print(pd.crosstab(df["Referred a Friend"], df["Number of Referrals"] > 0), "\n")

print("Internet Service vs Internet Type")
print(pd.crosstab(df["Internet Service"], df["Internet Type"]))

Under 30 vs (Age < 30)
Age       False  True 
Under 30              
No         5642      0
Yes           0   1401 

Senior Citizen vs (Age >= 65)
Age             False  True 
Senior Citizen              
No               5901      0
Yes                 0   1142 

Dependents vs (Number of Dependents > 0)
Number of Dependents  False  True 
Dependents                        
No                     5416      0
Yes                       0   1627 

Referred a Friend vs (Number of Referrals > 0)
Number of Referrals  False  True 
Referred a Friend                
No                    3821      0
Yes                      0   3222 

Internet Service vs Internet Type
Internet Type     Cable   DSL  Fiber Optic  No Internet Service
Internet Service                                               
No                    0     0            0                 1526
Yes                 830  1652         3035                    0


In [11]:
# 5.2 Total Revenue is an exact accounting identity of the other financial columns
calculated = (df["Total Charges"] - df["Total Refunds"]
              + df["Total Extra Data Charges"] + df["Total Long Distance Charges"])

print("Largest difference between Total Revenue and the recomputed value:",
      (calculated - df["Total Revenue"]).abs().max())

Largest difference between Total Revenue and the recomputed value: 1.8189894035458565e-12


In [12]:
# 5.3 Strongly correlated numerical pairs (|r| > 0.7) among the remaining numeric columns
num = df.select_dtypes(include=[np.number]).drop(columns=["Churn Score", "Satisfaction Score"])
corr = num.corr()

print("Correlated pairs (|r| > 0.7):")
for i, a in enumerate(corr.columns):
    for b in corr.columns[i + 1:]:
        if abs(corr.loc[a, b]) > 0.7:
            print(f"  {a:<30} {b:<30} r = {corr.loc[a, b]:.3f}")

Correlated pairs (|r| > 0.7):
  Zip Code                       Latitude                       r = 0.895
  Zip Code                       Longitude                      r = -0.791
  Latitude                       Longitude                      r = -0.886
  Tenure in Months               Total Charges                  r = 0.826
  Tenure in Months               Total Revenue                  r = 0.853
  Total Charges                  Total Revenue                  r = 0.972
  Total Long Distance Charges    Total Revenue                  r = 0.779


### 5.4 Redundancy decisions

| Removed | Kept instead | Justification |
|---|---|---|
| `Under 30` | `Age` | `Under 30` is exactly `Age < 30`; the crosstab has zero off-diagonal cases. `Age` keeps the full detail. |
| `Senior Citizen` | `Age` | Exactly `Age >= 65`. Same reasoning. |
| `Dependents` | `Number of Dependents` | Exactly `Number of Dependents > 0`; the count is strictly more informative. |
| `Referred a Friend` | `Number of Referrals` | Exactly `Number of Referrals > 0`. |
| `Internet Service` | `Internet Type` | After Member 1's cleaning, `Internet Type` already carries the level `No Internet Service` for every customer without internet, so the binary flag adds nothing. |
| `Total Revenue` | the individual charge columns | Exact accounting identity (difference ≈ 1e-12): Charges − Refunds + Extra Data + Long Distance. r = 0.97 with `Total Charges`. |
| `Total Charges` | `Tenure in Months` + `Monthly Charge` | `Total Charges` is essentially tenure × monthly charge (r = 0.83 with tenure, r = 0.65 with monthly charge). The two kept columns express the same information in a form that is not confounded by how long the customer has been with the company. |
| `Total Long Distance Charges` | `Avg Monthly Long Distance Charges` | r = 0.67 with tenure, so the total mostly measures tenure again. The monthly average measures actual usage intensity. |

`Total Refunds` and `Total Extra Data Charges` are kept: their correlation with every other numeric column
is below 0.15, so they contribute independent information even though their individual correlation with
churn is small.

## 6. Feature engineering

Only features with a clear business meaning are created, and each one is evaluated against the target
before it is accepted.

In [13]:
# 6.1 Total Services - how many services the customer actually subscribes to
service_cols = [
    "Phone Service", "Multiple Lines", "Internet Service", "Online Security", "Online Backup",
    "Device Protection Plan", "Premium Tech Support", "Streaming TV", "Streaming Movies",
    "Streaming Music", "Unlimited Data"
]

df["Total Services"] = (df[service_cols] == "Yes").sum(axis=1)

print(df["Total Services"].describe().round(2))

count    7043.00
mean        5.17
std         2.89
min         1.00
25%         3.00
50%         5.00
75%         7.00
max        11.00
Name: Total Services, dtype: float64


In [14]:
# 6.2 Evaluate Total Services against churn
svc = pd.crosstab(df["Total Services"], df[TARGET])
svc["churn_rate_%"] = (svc["Yes"] / (svc["Yes"] + svc["No"]) * 100).round(1)
print(svc)
print()
print("Linear correlation with target:", round(np.corrcoef(df["Total Services"], y)[0, 1], 3))

Churn Label       No  Yes  churn_rate_%
Total Services                         
1               1086  105           8.8
2                407   74          15.4
3                274  242          46.9
4                436  322          42.5
5                507  285          36.0
6                522  278          34.8
7                525  229          30.4
8                565  178          24.0
9                410  115          21.9
10               285   33          10.4
11               157    8           4.8

Linear correlation with target: 0.019


**Verdict — keep `Total Services`.**

The linear correlation is almost zero (0.019), but the crosstab shows the relationship is **U-shaped**, not
linear: customers with 1 service churn at 8.8%, the risk peaks at 46.9% for 3 services, and falls back to
4.8% for 11 services. Customers with a single product have little to leave behind, deeply bundled customers
are locked in, and the partially-bundled middle is the vulnerable group. A correlation coefficient cannot
see this shape, but tree-based models (Decision Tree, Random Forest) can split on it, so the feature is a
genuine addition rather than a restatement of the individual service flags.

Note that `Internet Service` is used to build this count before it is dropped as a redundant column.

In [15]:
# 6.3 Tenure Group - lifecycle stages rather than raw months
df["Tenure Group"] = pd.cut(
    df["Tenure in Months"],
    bins=[0, 12, 24, 48, 72],
    labels=["0-12", "13-24", "25-48", "49-72"],
    include_lowest=True
)

print(df["Tenure Group"].value_counts().sort_index())

Tenure Group
0-12     2186
13-24    1024
25-48    1594
49-72    2239
Name: count, dtype: int64


In [16]:
# 6.4 Evaluate Tenure Group against churn
tg = pd.crosstab(df["Tenure Group"], df[TARGET])
tg["churn_rate_%"] = (tg["Yes"] / (tg["Yes"] + tg["No"]) * 100).round(1)
print(tg)

Churn Label     No   Yes  churn_rate_%
Tenure Group                          
0-12          1149  1037          47.4
13-24          730   294          28.7
25-48         1269   325          20.4
49-72         2026   213           9.5


**Verdict — keep `Tenure Group`.**

Churn falls steeply across the lifecycle stages: 47.4% in the first year, 28.7% in the second, 20.4% up to
four years and 9.5% afterwards. The first-year risk is roughly five times the loyal-customer risk, which is
the clearest business story in the dataset.

An honest limitation to state in the viva: `Tenure Group` is a monotone binning of `Tenure in Months`, so it
adds **no new information for a tree model**, which can find those thresholds itself. Its value is for
scale-sensitive linear models such as Logistic Regression, where the raw tenure coefficient assumes a
straight-line effect while the grouped version lets each stage carry its own weight. Both are kept so that
Member 3 and the modelling stage can compare them; if the Evaluation 2 experiments show no gain,
`Tenure Group` should be dropped.

### Features considered and rejected

| Considered | Rejected because |
|---|---|
| `Avg Charge per Month` = Total Charges / Tenure | Nearly identical to the existing `Monthly Charge` column, so it is pure duplication. |
| `Revenue per Month` = Total Revenue / Tenure | Built from `Total Revenue`, which is already removed as an accounting identity. |
| `High CLTV` flag (CLTV above the median) | Throws away detail that the raw `CLTV` column already provides, and the threshold would be arbitrary. |

## 7. Apply the decisions and build the final feature set

In [17]:
redundant_cols = [
    "Under 30", "Senior Citizen", "Dependents", "Referred a Friend", "Internet Service",
    "Total Charges", "Total Revenue", "Total Long Distance Charges"
]

removed_cols = leakage_cols + constant_cols + geo_cols + redundant_cols

removal_log = pd.DataFrame(
    [(c, "Identifier / leakage") for c in leakage_cols]
    + [(c, "Zero variance") for c in constant_cols]
    + [(c, "Geographic: high cardinality / no signal") for c in geo_cols]
    + [(c, "Redundant with a retained column") for c in redundant_cols],
    columns=["Removed column", "Reason"]
)

print(removal_log.to_string(index=False))
print()
print("Total columns removed:", len(removed_cols))

             Removed column                                   Reason
                Customer ID                     Identifier / leakage
            Customer Status                     Identifier / leakage
             Churn Category                     Identifier / leakage
               Churn Reason                     Identifier / leakage
                Churn Score                     Identifier / leakage
         Satisfaction Score                     Identifier / leakage
                    Country                            Zero variance
                      State                            Zero variance
                    Quarter                            Zero variance
                       City Geographic: high cardinality / no signal
                   Zip Code Geographic: high cardinality / no signal
                   Latitude Geographic: high cardinality / no signal
                  Longitude Geographic: high cardinality / no signal
                 Population Geogra

In [18]:
final_df = df.drop(columns=removed_cols)

X = final_df.drop(columns=[TARGET])
y_final = final_df[TARGET]

print("Columns in cleaned_data.csv :", df.shape[1] - 2)
print("Columns removed             :", len(removed_cols))
print("Columns engineered          : 2  (Total Services, Tenure Group)")
print("Final X columns             :", X.shape[1])
print("Target                      :", TARGET, "|", y_final.shape[0], "rows")

Columns in cleaned_data.csv : 50
Columns removed             : 22
Columns engineered          : 2  (Total Services, Tenure Group)
Final X columns             : 29
Target                      : Churn Label | 7043 rows


## 8. Final X-column specification for Member 3

Member 3 needs two lists to build the `ColumnTransformer`: which columns go through
`SimpleImputer(median) + StandardScaler`, and which go through `SimpleImputer(most_frequent) +
OneHotEncoder`.

In [19]:
numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [c for c in X.columns if c not in numeric_features]

print("numeric_features =", numeric_features)
print()
print("categorical_features =", categorical_features)
print()
print(f"{len(numeric_features)} numeric + {len(categorical_features)} categorical = {X.shape[1]} features")

numeric_features = ['Age', 'Number of Dependents', 'Number of Referrals', 'Tenure in Months', 'Avg Monthly Long Distance Charges', 'Avg Monthly GB Download', 'Monthly Charge', 'Total Refunds', 'Total Extra Data Charges', 'CLTV', 'Total Services']

categorical_features = ['Gender', 'Married', 'Offer', 'Phone Service', 'Multiple Lines', 'Internet Type', 'Online Security', 'Online Backup', 'Device Protection Plan', 'Premium Tech Support', 'Streaming TV', 'Streaming Movies', 'Streaming Music', 'Unlimited Data', 'Contract', 'Paperless Billing', 'Payment Method', 'Tenure Group']

11 numeric + 18 categorical = 29 features


In [20]:
# One-hot width check, so Member 3 knows the size of the encoded matrix in advance
total_levels = sum(X[c].nunique() for c in categorical_features)

print("Categorical levels per column:")
for c in categorical_features:
    print(f"  {c:<28} {X[c].nunique()} levels")
print()
print("Approximate width after one-hot encoding:", len(numeric_features) + total_levels, "columns")

Categorical levels per column:
  Gender                       2 levels
  Married                      2 levels
  Offer                        6 levels
  Phone Service                2 levels
  Multiple Lines               2 levels
  Internet Type                4 levels
  Online Security              2 levels
  Online Backup                2 levels
  Device Protection Plan       2 levels
  Premium Tech Support         2 levels
  Streaming TV                 2 levels
  Streaming Movies             2 levels
  Streaming Music              2 levels
  Unlimited Data               2 levels
  Contract                     3 levels
  Paperless Billing            2 levels
  Payment Method               3 levels
  Tenure Group                 4 levels

Approximate width after one-hot encoding: 57 columns


In [21]:
# Final sanity checks before handover
assert TARGET in final_df.columns, "Target column is missing"
assert not any(c in X.columns for c in leakage_cols), "A leakage column survived into X"
assert X.isnull().sum().sum() == 0, "X still contains missing values"
assert len(X) == 7043, "Row count changed - no rows should be dropped at this stage"

print("All checks passed.")
print("Missing values in X :", X.isnull().sum().sum())
print("Duplicate columns   :", X.columns.duplicated().sum())

All checks passed.


Missing values in X : 0
Duplicate columns   : 0


## 9. Save the handover dataset

In [22]:
final_df.to_csv("feature_selected_data.csv", index=False)

print("Saved feature_selected_data.csv")
print("Shape:", final_df.shape, "=", X.shape[1], "features + 1 target")
print()
print(final_df.head())

Saved feature_selected_data.csv


Shape: (7043, 30) = 29 features + 1 target

   Gender  Age Married  Number of Dependents  Number of Referrals  \
0    Male   78      No                     0                    0   
1  Female   74     Yes                     1                    1   
2    Male   71      No                     3                    0   
3  Female   78     Yes                     1                    1   
4  Female   80     Yes                     1                    1   

   Tenure in Months     Offer Phone Service  \
0                 1  No Offer            No   
1                 8   Offer E           Yes   
2                18   Offer D           Yes   
3                25   Offer C           Yes   
4                37   Offer C           Yes   

   Avg Monthly Long Distance Charges Multiple Lines  ...        Contract  \
0                               0.00             No  ...  Month-to-Month   
1                              48.85            Yes  ...  Month-to-Month   
2                            

## 10. Member 2 deliverables — summary for Evaluation 1

### Final feature list (29 features)

| Group | Features |
|---|---|
| **Demographic** | Gender, Age, Married, Number of Dependents |
| **Tenure / relationship** | Tenure in Months, Number of Referrals, *Tenure Group* |
| **Service usage** | Phone Service, Multiple Lines, Internet Type, Avg Monthly GB Download, Avg Monthly Long Distance Charges, Online Security, Online Backup, Device Protection Plan, Premium Tech Support, Streaming TV, Streaming Movies, Streaming Music, Unlimited Data, *Total Services* |
| **Contract / billing** | Contract, Offer, Paperless Billing, Payment Method, Monthly Charge, Total Refunds, Total Extra Data Charges |
| **Customer value** | CLTV |

*Italic = engineered in this notebook.*

### Removed-feature / leakage table (22 columns removed)

| Reason | Columns |
|---|---|
| Identifier | Customer ID |
| Target leakage | Customer Status, Churn Category, Churn Reason, Churn Score, Satisfaction Score |
| Zero variance | Country, State, Quarter |
| Geographic (high cardinality / no signal) | City, Zip Code, Latitude, Longitude, Population |
| Redundant recodes | Under 30, Senior Citizen, Dependents, Referred a Friend, Internet Service |
| Redundant financial | Total Charges, Total Revenue, Total Long Distance Charges |

### Feature-selection rationale

1. Removed the identifier and every column that exists only because the outcome is already known.
2. Removed zero-variance columns, which cannot separate the classes.
3. Reviewed the geographic columns and removed them: 1,106 cities and 1,626 zip codes would explode the
   encoded feature space, and no location column correlates with churn beyond |r| = 0.06.
4. Reviewed the numerical correlation matrix and removed one column from each redundant pair, keeping the
   version that is not confounded by tenure.
5. Kept the customer, service, contract and billing variables that a telco genuinely knows about an active
   customer.
6. Engineered two features and tested each against the target before accepting it.

### Handover to Member 3

* File: `feature_selected_data.csv` — 7,043 rows × 30 columns (29 features + `Churn Label`).
* `Churn Label` is still stored as `Yes` / `No`; Member 3 maps it to 1 / 0, or passes it directly to
  `train_test_split(..., stratify=y)`, which accepts string labels.
* `numeric_features` and `categorical_features` are printed in section 8 and should be copied into the
  `ColumnTransformer`.
* No rows were dropped and no missing values were introduced, so Member 3's imputers will not change any
  value on this dataset — they stay in the pipeline to protect against unseen production data.

### Viva preparation — Member 2

* **What is data leakage?** Using information during training that would not be available at prediction
  time. The model looks excellent in testing and fails in production, because the leaking column does not
  exist for a customer who has not churned yet.
* **Why exclude Churn Reason / Category / Score?** Reason and Category are filled in only for customers who
  already left (5,174 blanks, all of them non-churners). Churn Score is a risk score computed from churn
  information, not a raw attribute of the customer.
* **Why exclude Customer Status?** Its crosstab with the target is perfectly separated: Churned → Yes,
  Joined/Stayed → No. It *is* the target.
* **Why exclude Satisfaction Score?** Scores 1–2 are 100% churners and scores 4–5 are 0% churners. It
  records how the relationship ended rather than predicting how it will end.
* **Which features were selected?** The 29 listed above — demographics, tenure and referrals, subscribed
  services, contract and billing behaviour, and CLTV.
* **What feature engineering was performed and why?** `Total Services` (breadth of the bundle — churn peaks
  at 46.9% for mid-bundle customers and falls at both ends, a shape no single service flag shows) and
  `Tenure Group` (lifecycle stages — 47.4% churn in year one against 9.5% after four years).